You can start from the code you already have and organise it into a fresh
notebook that does everything in one place, with the outlier‑mask‑and‑fill
step applied as soon as the numeric columns exist.

Below is the skeleton of **`merge_into_15min.ipynb`**; each
`# %%` marker denotes a separate cell.



In [ ]:
# %%  (first cell)
# bare‑bones imports and constants
import os
from pathlib import Path
import re

import numpy as np
import pandas as pd
from calendar import monthrange

# directories containing the raw workbooks
# BASE = Path(r"C:\Users\91930\Desktop\deep learning\2025 work\2026 merging\data")
# BASE = Path(r"C:\Users\91930\Desktop\SLDC complete\data")
BASE = Path(r"C:\Users\91930\Desktop\SLDC complete\Notebooks\Merging\data_raw")
SCADA_DIR = BASE / "scada"
REPORT_DIR = BASE / "report"

# columns that will be merged later
DEMAND_COLS = [
    "TPCODL Demand",
    "TPWODL Demand",
    "TPNODL Demand",
    "TPSOSDL Demand",
    "Total Demand (as recorded)",
]

# which columns we will never treat as demand outliers
EXCLUDE_COLS = {
    "Renewables Generation (SOLAR)",
    "Frequency",
    "Vedanta Drawl",
    "Time",        # numeric coercion makes this all NaN / out‑of‑range
}

LOW, HIGH = 0, 10000           # initial bounds for the five demand cols

In [ ]:
# %%  (cell 2)
# utility routines for inventory
MONTH_MAP = {
    "jan": 1, "january": 1,
    "feb": 2, "february": 2,
    "mar": 3, "march": 3,
    "apr": 4, "april": 4,
    "may": 5,
    "jun": 6, "june": 6,
    "jul": 7, "july": 7,
    "aug": 8, "august": 8,
    "sept": 9, "sep": 9, "september": 9,
    "oct": 10, "october": 10,
    "nov": 11, "november": 11,
    "dec": 12, "december": 12,
}

def extract_year_month(name: str):
    name = name.lower()
    year_m = re.search(r"20\d{2}", name)
    year = int(year_m.group()) if year_m else None
    month_word = next((w for w in MONTH_MAP if w in name), None)
    month = MONTH_MAP[month_word] if month_word else None
    return year, month

def build_inventory(folder: Path, source: str) -> pd.DataFrame:
    rows = []
    for f in folder.iterdir():
        if f.suffix.lower() in {".xls", ".xlsx"}:
            y, m = extract_year_month(f.name)
            rows.append({
                "source": source,
                "file_name": f.name,
                "file_path": str(f),
                "year": y,
                "month": m,
            })
    return pd.DataFrame(rows)

In [ ]:
# %%  (cell 3)
# make and inspect the DataFrame that lists all the files we will read
scada_df = build_inventory(SCADA_DIR, "scada")
report_df = build_inventory(REPORT_DIR, "report")
files_df = pd.concat([scada_df, report_df], ignore_index=True)
files_df = files_df.sort_values(["year", "month"])
files_df.head()

In [ ]:
# %%  (cell 4)
# optional: check the raw workbooks for extreme values in the demand cols
def find_initial_outliers(path,
                          low=LOW, high=HIGH,
                          include=DEMAND_COLS,
                          exclude=EXCLUDE_COLS):
    df = pd.read_excel(path)
    num = df.apply(lambda s: pd.to_numeric(s, errors="coerce"))
    if exclude:
        num = num.drop(columns=[c for c in exclude if c in num.columns],
                        errors="ignore")
    if include:
        num = num[[c for c in include if c in num.columns]]
    mask = (num < low) | (num > high)
    return mask

for _, r in files_df.iterrows():
    m = find_initial_outliers(r["file_path"])
    if m.any().any():
        cnt = m.sum()
        print(f"{r['file_name']} → {cnt.sum()} outlier(s)",
              cnt[cnt>0].to_dict())

In [ ]:
# %%  (cell 5)
# datetime parsing helpers
def parse_report_datetime(df: pd.DataFrame) -> pd.Series:
    date_col = "Date" if "Date" in df.columns else "DATE"
    if pd.api.types.is_datetime64_any_dtype(df["Time"]):
        return df["Time"]
    if pd.api.types.is_numeric_dtype(df["Time"]):
        return pd.to_datetime(df["Time"], unit="d",
                              origin="1899-12-30")
    if df["Time"].astype(str).str.contains("-").any():
        return pd.to_datetime(df["Time"], errors="coerce")
    df[date_col] = df[date_col].ffill()
    return pd.to_datetime(
        df[date_col].astype(str) + " " + df["Time"].astype(str),
        dayfirst=True, errors="coerce"
    )

def read_excel_clean(path: str) -> pd.DataFrame:
    df = pd.read_excel(path)
    df.columns = df.columns.astype(str).str.strip()
    return df

In [ ]:
# %%  (cell 6)
# read every workbook, add a datetime and concatenate
def read_all_files(files: pd.DataFrame) -> pd.DataFrame:
    all_dfs = []
    for _, r in files.iterrows():
        print("Processing", r["file_name"])
        df = read_excel_clean(r["file_path"])
        y, m = r["year"], r["month"]
        if r["source"] == "report":
            df["datetime"] = parse_report_datetime(df)
            df = df.dropna(subset=["datetime"])
        else:   # scada
            start = pd.Timestamp(year=y, month=m, day=1)
            days = monthrange(y, m)[1]
            full = pd.date_range(start=start, periods=days*1440, freq="min")
            df["datetime"] = full[:len(df)]
        all_dfs.append(df)
    master = pd.concat(all_dfs, ignore_index=True)
    master = master.sort_values("datetime").set_index("datetime")
    return master

master_df = read_all_files(files_df)

In [ ]:
# %%  (cell 7)
# keep only the columns we care about and convert to numeric
keep = (["datetime"] + DEMAND_COLS +
        ["Renewables Generation (SOLAR)", "Frequency", "Vedanta Drawl"])
master_clean = master_df.reset_index()[keep].copy()
master_clean = master_clean.rename(columns={"datetime": "Timestamp"})

for col in master_clean.columns.drop("Timestamp"):
    master_clean[col] = (
        master_clean[col].astype(str)
                         .str.replace(",", "")
                         .str.strip()
    )
    master_clean[col] = pd.to_numeric(master_clean[col], errors="coerce")

In [ ]:
# %%  (cell 8)
# replace any demand values outside the initial bounds with NaN
mask = ~master_clean[DEMAND_COLS].between(LOW, HIGH)
master_clean.loc[mask] = np.nan

# confirm the operation
master_clean[DEMAND_COLS].isna().sum()

In [ ]:
# %%  (cell 9)
# final tidy‑up: set timestamp index and save CSVs
master_clean["Timestamp"] = pd.to_datetime(master_clean["Timestamp"])
master_clean = master_clean.set_index("Timestamp")

master_clean.to_csv("master_1min_dataset.csv", index=False)
df_15min = master_clean.resample("15min").mean()
df_15min.to_csv("master_15min_dataset.csv")

In [ ]:
# %%  (cell 10)
# you can inspect the results interactively
master_clean.head()





















---

Copy the above cells into a new notebook named `merge_data_with_outlier_removal.ipynb`
and run them in sequence.  The logic is exactly the same as your original
notebook, but organised so that outliers are detected and turned to `NaN`
as part of the cleaning pipeline.  
Feel free to tweak the bounds, the `DEMAND_COLS` list, or the initial
inspection step to suit your needs.

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from calendar import monthrange

# --- CONFIGURATION ---
BASE_PATH = Path(r"C:\Users\91930\Desktop\deep learning\2025 work\2026 merging\data")
DEMAND_COLS = [
    "TPCODL Demand", "TPWODL Demand", "TPNODL Demand", 
    "TPSOSDL Demand", "Total Demand (as recorded)"
]
OTHER_COLS = ["Renewables Generation (SOLAR)", "Frequency", "Vedanta Drawl"]
ALL_COLS = DEMAND_COLS + OTHER_COLS
LOW_BOUND, HIGH_BOUND = 0, 10000

MONTH_MAP = {
    "jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
    "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12
}

# --- HELPER FUNCTIONS ---

def extract_meta(filename):
    """Extracts year and month index from filename."""
    name = filename.lower()
    year_match = re.search(r'20\d{2}', name)
    year = int(year_match.group()) if year_match else None
    
    month = None
    for word, idx in MONTH_MAP.items():
        if word in name:
            month = idx
            break
    return year, month

def parse_datetime(df, year, month, source):
    """Standardizes datetime across different file formats."""
    if source == "report":
        date_col = "Date" if "Date" in df.columns else "DATE"
        # Handle Excel Serial Numbers or Strings
        if pd.api.types.is_numeric_dtype(df["Time"]):
            df["datetime"] = pd.to_datetime(df["Time"], unit="d", origin="1899-12-30")
        else:
            df[date_col] = df[date_col].ffill()
            df["datetime"] = pd.to_datetime(
                df[date_col].astype(str) + " " + df["Time"].astype(str), 
                dayfirst=True, errors="coerce"
            )
    else:
        # SCADA logic: Generate minute-wise index for the month
        start_date = pd.Timestamp(year=year, month=month, day=1)
        days = monthrange(year, month)[1]
        df["datetime"] = pd.date_range(start=start_date, periods=len(df), freq="min")
    
    return df.dropna(subset=["datetime"])

# --- CORE LOGIC ---

def process_pipeline():
    # 1. Build Inventory
    files = []
    for folder, source in [("scada", "scada"), ("report", "report")]:
        for f in (BASE_PATH / folder).glob("*.xls*"):
            year, month = extract_meta(f.name)
            files.append({"path": f, "year": year, "month": month, "source": source})
    
    inventory_df = pd.DataFrame(files).sort_values(["year", "month"])
    
    # 2. Load and Clean
    all_data = []
    for _, row in inventory_df.iterrows():
        print(f"Reading: {row['path'].name}")
        df = pd.read_excel(row["path"])
        df.columns = df.columns.astype(str).str.strip()
        
        # Date Standardization
        df = parse_datetime(df, row["year"], row["month"], row["source"])
        
        # Select and Convert Numeric Columns
        valid_cols = [c for c in ALL_COLS if c in df.columns]
        for col in valid_cols:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", ""), errors="coerce")
            
            # Outlier handling (Demand only)
            if col in DEMAND_COLS:
                df.loc[(df[col] < LOW_BOUND) | (df[col] > HIGH_BOUND), col] = np.nan
        
        all_data.append(df[["datetime"] + valid_cols])

    # 3. Merge & Global Interpolation
    master = pd.concat(all_data).sort_values("datetime").set_index("datetime")
    
    # Fill gaps (Outliers or missing minutes)
    master[ALL_COLS] = master[ALL_COLS].ffill().bfill()
    
    # 4. Exports
    print("Exporting 1-min data...")
    master.to_csv("master_1min_dataset.csv")
    
    print("Exporting 15-min data...")
    master.resample("15min").mean().to_csv("master_15min_dataset.csv")
    
    print("Done!")

if __name__ == "__main__":
    process_pipeline()

Explanation of the Logic
1. Inventory & Metadata Management
Instead of hardcoding file lists, the script uses pathlib to scan your folders. The extract_meta function uses Regular Expressions (Regex) to find the year (e.g., "2021") and a dictionary map to convert month names (Jan, January) into integers (1). This ensures the files are merged in the correct chronological order.

2. The "Dual-Source" DateTime Strategy
The biggest challenge in your data is that SCADA and Report files handle time differently:

Report Files: These usually have a Date and Time column. The script handles "Excel Serial Numbers" (where 12:00 PM is stored as 0.5) and standard string formats.

SCADA Files: These often lack a full date. The script calculates how many minutes are in that specific month (using monthrange) and generates a synthetic timestamp starting from the 1st of that month.

3. Data Integrity & Cleaning
Outlier Removal: For power demand, values below 0 or above 10,000 are physically unlikely in this context. The script sets these to NaN (Not a Number).

Refined Imputation: Instead of leaving holes where outliers were removed, the script uses .ffill() (Forward Fill). If a data point is missing at 10:01 AM, it takes the value from 10:00 AM.

Numeric Coercion: It strips commas and whitespace from strings before converting to floats, preventing "Unable to parse string" errors.

4. Resampling
The final step creates two versions of the truth:

1-Minute: The raw, cleaned granular data.

15-Minute: The standard settlement period for power grids, calculated using the mean of the 1-minute blocks.

In [ ]:
## Mergin deamand data from 2017 to 2020 with the newly created master dataset

import pandas as pd

# 1. Load the historical 15-min data
historical_df = pd.read_csv('../Merging and cleaning/data_to_check/demand.csv')
historical_df["Timestamp"] = pd.to_datetime(historical_df["Timestamp"])
historical_df.set_index("Timestamp", inplace=True)

# 2. Filter for the 2017-2020 range (just to be safe)
historical_df = historical_df.loc['2017-01-01':'2020-12-31']

# 3. Concatenate with your current master_clean (assumed 15-min)
# 'axis=0' stacks them vertically
master_combined = pd.concat([historical_df, master_clean], axis=0)

# 4. Sort by the index to ensure chronological order
master_combined.sort_index(inplace=True)

# 5. Remove any accidental duplicates (e.g., if the files overlapped)
master_combined = master_combined[~master_combined.index.duplicated(keep='first')]

In [ ]:
# Create a perfect 15-minute range from start to finish
full_range = pd.date_range(start=master_combined.index.min(), 
                           end=master_combined.index.max(), 
                           freq='15T')

# Reindex to catch any missing time slots
master_combined = master_combined.reindex(full_range)

# # Fill any new gaps created by the reindexing
# master_combined = master_combined.ffill().bfill()

# master_combined.to_csv("master_combined_15min.csv", index=True, date_format="%Y-%m-%d %H:%M:%S")